# Mapowanie sprawozdań na standard biznesradar

Zamienia sprawozdanie spółki giełdowej na kolumnę w standardowym nazewnictwie
(`total_assets`, `revenues`, `ebit`, ...). Obsługuje MSSF i ustawę o rachunkowości,
spółki przemysłowe i banki, raporty w złotych, tysiącach, milionach i w walucie obcej.

**Stan słownika: 38 konfiguracji, 3 063 aliasy, 10 reguł obliczanych, 2 szablony standardu.**

### Co wgrać na początek

| plik | rola |
|---|---|
| `brmap.py` | cała logika — parser, mapowanie, walidacja, porównanie, kreator |
| `start.py` | program interaktywny |
| `slownik.xlsx` | **jedyny plik z wiedzą** — rośnie z każdą spółką |
| `test_slownik.py` | kontrola spójności słownika, nie wymaga danych spółek |
| sprawozdanie spółki | `.xlsx` albo PDF — patrz punkt 3 |
| plik wzorcowy `TICKER.xlsx` | opcjonalny, do kontroli poprawności |

### Jak korzystać z notatnika

- **Punkt 2** — program interaktywny. Jedna komórka, pyta o wszystko po kolei. Najprostsze.
- **Punkty 3–7** — krok po kroku, gdy chcesz kontrolować każdy etap albo coś debugować.
- **Punkt 8** — regresja na wszystkich spółkach. Odpal po każdej zmianie w słowniku.

## 1. Konfiguracja

Wybierz `TRYB`:

| tryb | kiedy | słownik po zamknięciu sesji |
|---|---|---|
| `"github"` | **zalecany** — repo klonuje się **na Dysk**, więc `git pull` / `git push` działa i nic nie ginie | zostaje |
| `"dysk"` | pliki wgrane ręcznie na Dysk, bez gita | zostaje |
| `"upload"` | szybki jednorazowy test | **ginie** |

> Gdyby sklonować repo do `/content`, wszystkie zmiany w `slownik.xlsx` przepadłyby
> razem z sesją. Dlatego tryb `"github"` klonuje do folderu na Dysku.

In [ ]:
TRYB = "github"     # "github" | "dysk" | "upload"
REPO = "https://github.com/maciejkfrankowski/testowo.git"
BAZA = "/content/drive/MyDrive"      # folder na Dysku
KATALOG = "testowo"                  # nazwa folderu repo

import os, sys, subprocess
try:
    import openpyxl
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openpyxl"], check=True)

if TRYB in ("github", "dysk"):
    from google.colab import drive
    drive.mount("/content/drive")

if TRYB == "github":
    cel = os.path.join(BAZA, KATALOG)
    if os.path.isdir(os.path.join(cel, ".git")):
        os.chdir(cel)
        print("Repo juz jest na Dysku - pobieram zmiany:")
        print(subprocess.run(["git", "pull"], capture_output=True, text=True).stdout)
    else:
        os.makedirs(BAZA, exist_ok=True)
        os.chdir(BAZA)
        subprocess.run(["git", "clone", REPO, KATALOG], check=True)
        os.chdir(cel)
elif TRYB == "dysk":
    os.makedirs(os.path.join(BAZA, KATALOG), exist_ok=True)
    os.chdir(os.path.join(BAZA, KATALOG))
else:
    os.chdir("/content")
    print("Wgraj: brmap.py, start.py, slownik.xlsx oraz plik sprawozdania")
    from google.colab import files
    files.upload()

sys.path.insert(0, os.getcwd())
print("\nFolder roboczy:", os.getcwd())
print("Pliki:", sorted(f for f in os.listdir(".") if f.endswith((".py", ".xlsx"))))


In [ ]:
if not os.path.exists("brmap.py"):
    raise SystemExit("Brak brmap.py w folderze roboczym - sprawdz TRYB i sciezki w komorce wyzej.")

import importlib, brmap, start
importlib.reload(brmap); importlib.reload(start)

import openpyxl
_w = openpyxl.load_workbook("slownik.xlsx")
print("slownik.xlsx:",
      _w["Spolki"].max_row - 1, "spolek |",
      _w["Aliasy"].max_row - 1, "aliasow |",
      _w["Reguly_obliczane"].max_row - 1, "regul")
print("kody:", " ".join(str(r[0]) for r in _w["Spolki"].iter_rows(min_row=2, values_only=True) if r[0]))

## 2. Program interaktywny (najprostsza droga)

Zadaje pytania po kolei: skrót spółki, plik sprawozdania, podpisy okresów, plik wzorcowy.
Nieznaną spółkę obsługuje kreatorem i **czeka**, aż uzupełnisz słownik.
Efekt: `wynik_TICKER_OKRES.xlsx` — nazwa zawiera okres, więc kolejny raport nie nadpisze poprzedniego.

> W Colabie uruchamiaj **tylko tak, jak niżej**. `!python start.py` nie zadziała —
> podproces nie ma dostępu do klawiatury i program przerwie się z komunikatem.

In [ ]:
import start
start.main()      # folder roboczy ustawila komorka konfiguracyjna

## 3. Sprawozdanie w PDF

Ścieżka `.xlsx` (scrapowany albo z ESEF) jest najpewniejsza — nazwy pozycji są pełne.
Gdy masz PDF, najpierw sprawdź, **czym ten plik w ogóle jest**. To decyduje o dalszej drodze
i oszczędza godzinę zgadywania.

| co zwraca `pdftotext` | droga |
|---|---|
| sensowny tekst z liczbami | `pdf_na_xlsx.py`, a gdy układ jest nietypowy — konwerter per spółka |
| same nagłówki stron / pustka | strony są **obrazkami** — patrz niżej |

Konwertery per spółka (`jsw_conv.py`, `opl_conv.py`, `kruk_conv.py`, `pge_conv.py`, `asb_conv.py`,
`echo_conv.py`, `pzu_conv.py`, `f51_conv.py`, `mdk_conv.py`, `agl_conv.py`) mają na górze komentarz
mówiący, co było nietypowe w danym raporcie. Przy kolejnym raporcie tej samej spółki zwykle
wystarczy podmienić numery linii.

In [ ]:
PDF = "raport_spolki.pdf"      # <- wgraj plik do folderu roboczego

!pdfinfo "{PDF}" | head -20
!pdftotext -layout "{PDF}" /tmp/r.txt && wc -l /tmp/r.txt
!head -40 /tmp/r.txt

### 3a. Strony jako obrazki

Zdarza się na NewConnect (Madkom, Agroliga): sprawozdania są wklejonymi bitmapami,
`pdftotext` wyciąga z nich wyłącznie nagłówki stron. Wtedy renderujemy strony do PNG.

**Tesseract w Colabie ma domyślnie tylko `eng`.** Przy raporcie po angielsku OCR działa
i wystarczy weryfikacja arytmetyczna. Przy polskich etykietach doinstaluj `tesseract-ocr-pol`
albo przepisz tabelę ręcznie — zniekształcone diakrytyki psują dopasowanie aliasów bardziej,
niż pomaga automat.

> Zawsze sprawdzaj wynik OCR **łańcuchem arytmetycznym**, nie wzrokiem. U Agroligi OCR
> przeczytał `(306)` jako `(396)` — wyłapała to dopiero suma kontrolna rachunku wyników.

In [ ]:
OD, DO = 12, 15                # zakres stron ze sprawozdaniami

!pdfimages -list -f {OD} -l {OD} "{PDF}" | head -5
!pdftoppm -f {OD} -l {DO} -r 200 -png "{PDF}" /content/str
!ls -la /content/str-*.png

# OCR - tylko gdy raport jest po angielsku
!apt-get -qq install -y tesseract-ocr-pol   # odkomentuj przy polskich etykietach
for p in range(OD, DO + 1):
    !tesseract /content/str-{p}.png /content/o-{p} --psm 6 -l eng 2>/dev/null

print(open(f"/content/o-{OD}.txt", encoding="utf-8").read()[:2000])

In [ ]:
# Podglad strony w notatniku - do recznego przepisania albo kontroli OCR
from IPython.display import Image, display
display(Image(f"/content/str-{OD}.png", width=1000))

### 3b. Co potem

Konwerter — dowolny z powyższych albo nowy na wzór `agl_conv.py` — ma jedno zadanie:
zapisać `TICKER_OKRES_sprawozdania.xlsx` w układzie, który czyta `brmap`:

- kolumna 1: nazwa pozycji, kolumny 2+: wartości
- wiersze-nagłówki oddzielające sekcje (`Non-current assets`, `Cash flows from financing activities`, ...)
- nazwy pozycji **dokładnie takie jak w raporcie** — to one są kluczem aliasu

Markery sekcji wpisuje się potem w kolumnę `sekcje` arkusza `Spolki`:
`Naglowek=>KOD` otwiera blok, `$Suma=>KOD` zamyka (dla spółek, które nie dają nagłówków,
tylko sumy na końcu — jak KGHM).

## 4. Nowa spółka — kreator

Uruchom **tylko przy pierwszym** sprawozdaniu danej spółki. Przy kolejnych kwartałach
przejdź od razu do punktu 5.

Kreator wykrywa m.in. czy dane są w tysiącach czy milionach oraz czy sumy bilansowe
**otwierają** bloki (jak u Neuki), czy je **zamykają** (jak u KGHM).

In [ ]:
PLIK   = "raport_spolki.xlsx"     # <- plik sprawozdania (albo wynik konwertera z pkt. 3)
TICKER = "ABC"                     # <- skrot spolki
NAZWA  = "Nazwa Spolki"

brmap.zbadaj_plik(PLIK)            # podglad wykrytego ukladu, nic nie zapisuje

In [ ]:
brmap.nowa_spolka(PLIK, "slownik.xlsx", TICKER, NAZWA)

### Co zrobić z plikiem `propozycja_<TICKER>.xlsx`

Otwórz go i przejdź dwa arkusze:

**`1_Wiersz_do_Spolki`** — skopiuj ten wiersz do arkusza `Spolki` w `slownik.xlsx`.
Sprawdź `mnoznik`:

| mnożnik | dane w raporcie |
|---|---|
| `1` | tysiące |
| `1000` | miliony |
| `0.001` | złote (typowe dla ustawy o rachunkowości i NewConnect) |
| `3.7708`, `4.2267` | **kurs waluty z dnia bilansowego** — spółka raportuje w USD / EUR |

> Przy spółce walutowej (Asbis, Agroliga) mnożnik trzeba **podmieniać przy każdym raporcie**.
> Biznesradar przelicza kursem z dnia bilansowego, a nie średnim za okres — również rachunek
> wyników i przepływy. Najpewniejszy sposób wyznaczenia kursu: podzielić dowolną pozycję
> z ich kolumny przez tę samą pozycję z raportu i sprawdzić na kilkunastu innych.

**`3_Propozycje_aliasow`** — popraw kolumnę `standard_key`, potem skopiuj kolumny **A–I**
do arkusza `Aliasy`. Kolumny J, K, L są tylko do przeglądu.

| kolor | co znaczy |
|---|---|
| zielony | pewne, można zatwierdzić |
| żółty | prawdopodobne, sprawdź |
| pomarańczowy | słabe dopasowanie — podpowiedź, nie odpowiedź |
| czerwony | brak podpowiedzi, zmapuj ręcznie |

`standard_key = POMIN` oznacza świadome pominięcie wiersza. Każdy wiersz sprawozdania
musi mieć alias — albo prawdziwy klucz, albo `POMIN`. Arkusz `Niezmapowane` ma być pusty.

## 5. Mapowanie

Powtarzaj aż `Niezmapowanych: 0`, a walidacja pokaże komplet `OK`.

**Ten sam kod spółki dla wszystkich okresów.** Kod jest jednocześnie kluczem wyjątków —
założenie drugiego kodu (`OPN_2026`) dla innego okresu gubi wszystkie wyjątki spółki
i mapowanie rozjeżdża się, zwykle na znakach kosztów. Zmieniają się tylko podpisy kolumn,
a te podaje się parametrem `okresy`.

In [ ]:
# Podpisy kolumn. Trafiaja do naglowkow wyniku I do pola balance_date.
# None = wez z arkusza Spolki.
OKRESY = None
# OKRESY = ["I polrocze 2026 / 30.06.2026", "IV kw 2025 / 31.12.2025"]

# Bilans poprzedniego okresu - potrzebny, gdy druga kolumna sprawozdania to ten sam
# kwartal ROK WCZESNIEJ (typowe dla ustawy o rachunkowosci). Wtedy kapitalu obrotowego
# nie da sie policzyc z bilansu i w arkuszu Spolki stoi kapital_obrotowy_z_bilansu = NIE.
#   PREV = brmap.wczytaj_okres_wzorca(f"{TICKER}.xlsx", "2025-12-31")
PREV = None

out, wyniki, walid, niezmapowane = brmap.mapuj(
    "slownik.xlsx", TICKER, PLIK, bilans_poprzedni=PREV, okresy=OKRESY)

Wynik ma cztery arkusze:

- **Wynik** — kolumna gotowa do wklejenia do standardu
- **Walidacja** — sumy kontrolne (14 dla spółek zwykłych, 13 dla banków)
- **Audyt** — z którego wiersza raportu wzięła się każda liczba
- **Niezmapowane** — co jeszcze zostało do zrobienia

Ostatni wiersz walidacji to kontrola miękka: implikowana stopa podatku poza 0–40%
zwykle oznacza błąd mapowania wyniku. Przy stracie pokazuje `brak danych` — to **nie** jest błąd.

### Wariant układu — kolumna `wyjatki_z`

Gdy zmienia się nie podpis, ale **układ kolumn** — bo raport za III kwartał podaje osobno
kwartał i narastająco, a roczny tylko rok — załóż wiersz wariantu:

| spolka | kol_wartosci | kol_wartosci_bilans | wyjatki_z |
|---|---|---|---|
| `OPN` | `2,3` | | |
| `OPN_3Q` | `3,5` | `2,4` | `OPN` |

`wyjatki_z` sprawia, że wariant **dziedziczy wszystkie wyjątki spółki macierzystej**,
więc nie powtarza się ich w arkuszu `Aliasy`. Wariantów jest jeden na *układ*, nie na okres.

> Częsta pułapka: gdy rachunek wyników ma 4 kolumny, przepływy zwykle też.
> `kol_wartosci_rzis` ustawione bez poprawienia `kol_wartosci` daje cichą bzdurę
> — amortyzacja i przepływy operacyjne wzięte z kolumny kwartalnej zamiast narastającej.

## 6. Porównanie ze wzorcem

Jeśli spółka jest już w standardzie — **najskuteczniejszy sposób wyłapywania błędów**.
Przy każdej dotąd zmapowanej spółce wywracał założenie, które wyglądało na oczywiste.

In [ ]:
# tolerancja 2 dla spolek raportujacych w ZLOTYCH (mnoznik 0.001) - zaokraglenia daja +/-1
# tolerancja 3 dla spolek walutowych - zaokraglenia po przeliczeniu kursem
TOL = 2.0 if brmap.zbadaj_plik(PLIK)["mnoznik"] < 1 else 1.0

brmap.porownaj(f"{TICKER}.xlsx", "2026-03-31", out, tol=TOL)

### Gdy nie masz pliku wzorcowego

Kolumnę da się przeczytać wprost ze strony. Trzy adresy na spółkę:

```
https://www.biznesradar.pl/raporty-finansowe-bilans/TICKER,Y
https://www.biznesradar.pl/raporty-finansowe-rachunek-zyskow-i-strat/TICKER,Y
https://www.biznesradar.pl/raporty-finansowe-przeplywy-pieniezne/TICKER,Y
```

`,Y` roczne · `,Q` kwartalne · `,C` skumulowane · `,C,1..4` filtr kwartału.
Część tickerów wymaga pełnego sluga: `AGROLIGA-GROUP`, `JSW-JASTRZEBSKA-SPOLKA-WEGLOWA`,
`ECHO-INVESTMENT`, `THE-FARM-51-GROUP`, `ASBISC`, `ORANGE`.

Dwie rzeczy, o które łatwo się potknąć:

1. Kolumna „Kapitał własny" na **stronie** to kapitał akcjonariuszy jednostki dominującej,
   a **wzorzec** trzyma w `capital` kapitał łączny z mniejszościami. Różnica = `nonshare_capital`.
2. Dane kwartalne RZiS i CashFlow biznesradar **wylicza z danych skumulowanych** —
   porównuj narastająco, nie kwartał do kwartału.

## 7. Pobranie wyników

W trybach „github" i „dysk" pliki są już na Dysku — ta komórka przydaje się tylko przy „upload".

In [ ]:
from google.colab import files
for f in (out, out.replace("wynik_", "porownanie_")):
    if os.path.exists(f):
        files.download(f)

## 8. Regresja — po każdej zmianie w słowniku

Przemapowuje **wszystkie** spółki ze słownika i pokazuje, które sumy kontrolne nie przechodzą.
Odpal zawsze po dopisaniu reguły obliczanej albo aliasu globalnego (bez kodu spółki) —
jedno i drugie działa na wszystkie spółki naraz i potrafi po cichu zepsuć coś, co działało.

In [ ]:
import io, contextlib, openpyxl

SZUKAJ = [".", "/content", "/content/drive/MyDrive"]     # gdzie leza pliki sprawozdan

w = openpyxl.load_workbook("slownik.xlsx")
h = [c.value for c in w["Spolki"][1]]
wiersze = [dict(zip(h, r)) for r in w["Spolki"].iter_rows(min_row=2, values_only=True) if r[0]]

print(f"{'kod':8} {'sumy':>7} {'niezmap':>8}  uwagi")
print("-" * 72)
brak, zle = [], []
for c in wiersze:
    kod = c["spolka"]
    s = next((os.path.join(d, c["plik"]) for d in SZUKAJ
              if c["plik"] and os.path.exists(os.path.join(d, c["plik"]))), None)
    if not s:
        brak.append(kod); continue
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            brmap.mapuj("slownik.xlsx", kod, s, out=f"/tmp/reg_{kod}.xlsx", cicho=True)
    except Exception as e:
        print(f"{kod:8} {'BLAD':>7}           {e}"); zle.append(kod); continue
    v = openpyxl.load_workbook(f"/tmp/reg_{kod}.xlsx")
    testy = list(v["Walidacja"].iter_rows(min_row=2, values_only=True))
    # "brak danych" w kontroli stopy podatku to strata, nie blad
    bledy = [t for t in testy if str(t[1]).startswith(("ROZNICA", "PODEJRZANA"))]
    nz = v["Niezmapowane"].max_row - 1
    opis = "; ".join(f"{t[0]}: {t[1]}" for t in bledy)
    print(f"{kod:8} {len(testy)-len(bledy):>3}/{len(testy):<3} {nz:>8}  {opis}")
    if bledy or nz: zle.append(kod)

print()
print("bez pliku zrodlowego:", brak or "brak")
print("do sprawdzenia      :", zle or "brak")

### Znane, świadomie zostawione rozjazdy

| spółka | test | różnica | dlaczego |
|---|---|---|---|
| `ALR` | WNiP = firma + pozostałe | −574 602 | brak rozbicia goodwill / pozostałe WNiP — do wyciągnięcia z not |
| `TOA_R` | Przepływy razem | −24 278 | do wyjaśnienia |
| `MDK` | Aktywa trwałe + obrotowe = suma | −123 | udziały własne stoją po stronie **aktywów** (ustawa o rachunkowości); standard nie ma na to klucza, biznesradar ma identyczny rozjazd w każdym okresie |

Te trzy nie są regresją — jeśli w tabeli wyżej pojawi się cokolwiek innego, to jest.

## 9. Odesłanie zmian słownika do repo

Dotyczy tylko trybu `"github"`. Po dopisaniu aliasów warto odesłać `slownik.xlsx` na GitHub,
żeby zmiany nie zostały tylko na Twoim Dysku.

Komórka najpierw eksportuje CSV-ki (`git diff` nie powie nic o binarce) i uruchamia
kontrolę spójności słownika — literówka w `standard_key`, alias wpisany dwa razy albo
reguła odwołująca się do nieistniejącego klucza objawiają się potem jako cicha bzdura w wyniku.

GitHub nie przyjmuje zwykłego hasła — potrzebny jest **personal access token**
(GitHub → Settings → Developer settings → Personal access tokens → uprawnienie `repo`).
Wklej go w okienko, które pojawi się pod komórką; nie zapisuj go w kodzie notatnika.

In [ ]:
import getpass, subprocess

subprocess.run([sys.executable, "slownik_csv.py"], check=True)      # czytelne diffy
subprocess.run([sys.executable, "test_slownik.py"], check=True)     # kontrola spojnosci

In [ ]:
uzytkownik = input("Nazwa uzytkownika GitHub: ").strip()
token = getpass.getpass("Personal access token (nie bedzie widoczny): ").strip()
opis = input("Opis zmiany: ").strip() or "Aktualizacja slownika"

subprocess.run(["git", "config", "user.email", "mf@biznesradar.pl"], check=True)
subprocess.run(["git", "config", "user.name", uzytkownik], check=True)
subprocess.run(["git", "add", "slownik.xlsx", "slownik_csv"], check=True)
subprocess.run(["git", "commit", "-m", opis], check=False)

url = REPO.replace("https://", f"https://{uzytkownik}:{token}@")
subprocess.run(["git", "push", url], check=True)
del token
print("Wypchniete.")

---

## Ściąga

```python
brmap.zbadaj_plik(plik)                       # podglad wykrytego ukladu (arkusz, kolumny, mnoznik, sekcje)
brmap.nowa_spolka(plik, slownik, ticker)      # kreator konfiguracji i propozycji aliasow
brmap.mapuj(slownik, ticker, plik,            # mapowanie + walidacja
            okresy=[...], bilans_poprzedni=...)
brmap.porownaj(wzorzec, data, wynik, tol=...) # kontrola ze wzorcem
brmap.wczytaj_okres_wzorca(wzorzec, data)     # bilans poprzedniego okresu ze standardu
```

### Dwa parametry, o których łatwo zapomnieć

`okresy` — podpisy kolumn dla tego konkretnego raportu. Bez nich wynik dostanie nagłówki
z arkusza `Spolki`, czyli z poprzedniego kwartału, razem z błędną `balance_date`.

`bilans_poprzedni` — gdy druga kolumna sprawozdania to ten sam kwartał rok wcześniej
(układ ustawy o rachunkowości). Bez tego kapitał obrotowy policzy się z niewłaściwej bazy.

### Dwa szablony standardu

Kolumna `szablon` w arkuszu `Spolki`:

- **`RAP`** — spółki przemysłowe i handlowe, 77 kluczy, 14 sum kontrolnych
- **`RAP_B`** — banki, 74 własne klucze (`interest_income`, `net_fee_income`,
  `loans_to_customers`...), 13 sum kontrolnych. Bank nie ma kapitału obrotowego
  ani podziału na aktywa trwałe i obrotowe. Klucze bankowe leżą w arkuszu `Klucze_B`.

Ubezpieczyciel **nie** wymaga trzeciego szablonu — PZU siedzi na zwykłym `RAP`
(cztery linie przychodowe do `revenues`, dwanaście kosztowych do `cost_of_sales`).

### Ustalona metodologia

- **kapitał obrotowy** nie jest przepisywany z rachunku przepływów, tylko liczony z różnic
  kolejnych bilansów, **od końca poprzedniego ROKU** — przy I kwartale to to samo,
  przy II–IV już nie. Reguła obowiązuje od IV kwartału 2022
- `current_borrowings` / `noncurrent_borrowings` = **wyłącznie** linia „Kredyty i pożyczki";
  faktoring i „inne zobowiązania finansowe" idą w `*_other_liabilities`.
  Gdy spółka daje jedną linię „kredyty, obligacje i leasing" — całość w `*_borrowings`,
  a `*_obligations` zostaje zerowe (PGE, Echo)
- `property` zawiera prawo do użytkowania; `right_to_use_assets` to pozycja „w tym",
  która **nie** sumuje się do aktywów trwałych
- `current_investments` zawiera środki pieniężne; `assets_for_sale` to składnik aktywów obrotowych
- agio → `reserve` (JSW, Orange, Kruk, Asbis, Agroliga)
- w wariancie **porównawczym** całe „Koszty działalności operacyjnej" idą w `cost_of_sales`,
  a `distribution_expenses` i `administrative_expenses` zostają puste. W **kalkulacyjnym**
  każda z trzech linii ma własny klucz
- w ustawie o rachunkowości `reckoning` = rezerwy + rozliczenia międzyokresowe
- rachunek wyników i przepływy są **narastające**; IV kwartał = cały rok
- **raport roczny przekształca dane porównawcze**, standard trzyma bilans pierwotnie
  zaraportowany — przy raporcie rocznym dociągaj okres odniesienia z pliku wzorcowego
- `year_profit` wypełnia się tylko wtedy, gdy spółka wykazuje wynik okresu osobno w bilansie
- strata nie wymaga obsługi: klucze wynikowe wychodzą ujemne, koszty zostają dodatnie

Pełny opis w arkuszu `README` wewnątrz `slownik.xlsx` oraz w `znaleziska_biznesradar.xlsx`
(25 udokumentowanych rozbieżności po stronie biznesradaru).